In [1]:
%pip install pandas numpy 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np

In [ ]:
import numpy as np


class TreeNode:
    """
    A node in a decision tree.  

    Attributes
    ----------
    feature_index : int
        The index of the feature used to split the node
    threshold : float
        The threshold value used to split the node
    left : TreeNode
        The left child of the node
    right : TreeNode
        The right child of the node
    prediction : int
        The predicted class for the node
    is_leaf : bool
        Whether the node is a leaf node
    depth : int
        The depth of the node in the decision tree

    """

    def __init__(
        self,
        feature_index=None,
        threshold=None,
        left=None,
        right=None,
        prediction=None,
        is_leaf=False,
        depth=0
    ):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        self.is_leaf = is_leaf
        self.depth = depth


class DecisionTreeClassifier:
    """
    A decision tree classifier.

    Attributes
    ----------
    max_depth : int
        The maximum depth of the decision tree
    min_samples_split : int
        The minimum number of samples required to split an internal node
    min_samples_leaf : int
        The minimum number of samples required to be at a leaf node
    impurity_measure : str
        The impurity measure to use when splitting nodes
    """

    def __init__(
        self,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        impurity_measure='entropy'
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.impurity_measure = impurity_measure
        self.root = None

    def fit(self, X, y):
        """
        Build a decision tree from the training data.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            The training data
        y : array-like, shape (n_samples,)
            The target values

        Returns
        -------
        None
        """
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        """
        Build a decision tree from the training data.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            The training data
        y : array-like, shape (n_samples,)
            The target values
        depth : int
            The current depth of the tree

        Returns
        -------
        TreeNode
            The root node of the decision tree
        """
        n_samples = len(y)
        n_classes = len(np.unique(y))

        # Stopping conditions
        if (
            (self.max_depth is not None and depth >= self.max_depth) or
            n_classes == 1 or
            n_samples < self.min_samples_split
        ):
            return TreeNode(
                is_leaf=True,
                prediction=self._majority_class(y),
                depth=depth
            )

        impurity = self._impurity(y)

        best_feature, best_threshold, best_gain, left_mask, right_mask = \
            self._find_best_split(X, y, impurity)

        if best_feature is None or best_gain <= 0:
            return TreeNode(
                is_leaf=True,
                prediction=self._majority_class(y),
                depth=depth
            )

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return TreeNode(
            feature_index=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
            is_leaf=False,
            depth=depth
        )

    def _find_best_split(self, X, y, parent_impurity):
        """
        Find the best split for a node in the decision tree.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            The training data
        y : array-like, shape (n_samples,)
            The target values
        parent_impurity : float
            The impurity of the parent node

        Returns
        -------
        best_feature : int
            The index of the feature used to split the node
        best_threshold : float
            The threshold value used to split the node
        best_gain : float
            The information gain of the split
        best_left_mask : array-like, shape (n_samples,)
            The mask of the left child
        best_right_mask : array-like, shape (n_samples,)
            The mask of the right child
        """
        best_feature = None
        best_threshold = None
        best_gain = -1
        best_left_mask = None
        best_right_mask = None

        n_samples = len(y)

        for feature_index in range(X.shape[1]):
            feature_values = X[:, feature_index]
            thresholds = np.unique(feature_values)

            for thr in thresholds:
                left_mask = feature_values <= thr
                right_mask = feature_values > thr

                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)

                # Skip invalid splits
                if (
                    n_left == 0 or
                    n_right == 0 or
                    n_left < self.min_samples_leaf or
                    n_right < self.min_samples_leaf
                ):
                    continue

                left_impurity = self._impurity(y[left_mask])
                right_impurity = self._impurity(y[right_mask])

                w_impurity = (
                    (n_left / n_samples) * left_impurity +
                    (n_right / n_samples) * right_impurity
                )

                info_gain = parent_impurity - w_impurity

                if info_gain > best_gain:
                    best_feature = feature_index
                    best_threshold = thr
                    best_gain = info_gain
                    best_left_mask = left_mask
                    best_right_mask = right_mask

        return best_feature, best_threshold, best_gain, best_left_mask, best_right_mask

    def _impurity(self, y):
        """
        Calculate the impurity of a node in the decision tree.

        Parameters
        ----------
        y : array-like, shape (n_samples,)
            The target values

        Returns
        -------
        impurity : float
            The impurity of the node
        """
        if self.impurity_measure == 'entropy':
            return self._entropy(y)
        elif self.impurity_measure == 'gini':
            return self._gini(y)
        else:
            raise ValueError("Invalid impurity measure")

    def _entropy(self, y):
        """
        Calculate the entropy of a node in the decision tree.
        -Σ(p_i * log₂(p_i))

        Parameters
        ----------
        y : array-like, shape (n_samples,)
            The target values

        Returns
        -------
        entropy : float
            The entropy of the node
        """
        _, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        return -np.sum(probs * np.log2(probs))


    def _gini(self, y):
        """
        Calculate the gini impurity of a node in the decision tree.
        1 - Σ(p_i²)

        Parameters
        ----------
        y : array-like, shape (n_samples,)
            The target values

        Returns
        -------
        gini : float
            The gini impurity of the node
        """
        _, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        return 1 - np.sum(probs**2)

    def _majority_class(self, y):
        """
        Calculate the majority class of a node in the decision tree.

        Parameters
        ----------
        y : array-like, shape (n_samples,)
            The target values

        Returns
        -------
        majority_class : int
            The majority class of the node
        """
        return np.argmax(np.unique(y, return_counts=True)[1])

    def _predict_one(self, x, node):
        """ 
        Predict the class of a single sample using the decision tree.

        Parameters
        ----------
        x : array-like, shape (n_features,)
            The sample to predict
        node : TreeNode
            The current node in the decision tree
        
        Returns
        -------
        prediction : int
            The predicted class
        """
        if node.is_leaf:
            return node.prediction

        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        """
        Predict the class of multiple samples using the decision tree.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            The samples to predict

        Returns
        -------
        predictions : array-like, shape (n_samples,)
            The predicted classes
        """
        return np.array([self._predict_one(x, self.root) for x in X])

In [11]:
import numpy as np

X = np.array([
    [2],
    [3],
    [10],
    [12]
])

y = np.array([0, 0, 1, 1])

tree = DecisionTreeClassifier(max_depth=2)
tree.fit(X, y)

preds = tree.predict(X)
print("Predictions:", preds)
print("True:", y)

acc = np.mean(preds == y)
print("Accuracy:", acc)



Predictions: [0 0 1 1]
True: [0 0 1 1]
Accuracy: 1.0


In [12]:
X = np.array([[1], [2], [3]])
y = np.array([1, 1, 1])

tree = DecisionTreeClassifier()
tree.fit(X, y)

print(tree.predict(X))

[1 1 1]


In [13]:
tree = DecisionTreeClassifier(min_samples_leaf=3)
tree.fit(X, y)
print(tree.predict(X))

[1 1 1]
